# HCE v11 — Biologically Accurate One-Shot Cell-Type Prediction

Full HCE retraining with:
- CL-backed hybrid ontology (`multi_tissue_v11_config.build_combined_ontology`)
- Canonical merged leaf vocabulary (`CANONICAL_LABEL_MAP`, 285 leaves, 0 duplicate OPC variants)
- snRNA normalization fix for glioma + Brain_normal
- Brain max_cells equalized (300 per organ)
- Attractor down-weighting (S2-E)
- Glioma reported lineage-only


## 1. Imports

In [1]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.sparse as sp
import h5py

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology
from cell2sentence.hierarchy_utils import deduplicate_hierarchy
from multi_tissue_v11_config import (
    CANONICAL_LABEL_MAP, CURATED_OVERRIDES, LUNG_HIERARCHY_COLS,
    LUNG_ROOT_ANCHORS, build_combined_ontology, assign_lineage,
    is_valid_label, canonicalize,
)

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
print('=' * 60)


/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/c2s-justin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  IMPORTS OK
  PyTorch : 2.10.0+cu128
  CUDA    : True | devices: 1


## 2. Configuration

In [2]:
# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR          = 'multi_tissue_v11_results'
HCE_MODEL_PATH   = os.path.join(OUT_DIR, 'hce_v11_best_model.pt')
TEMP_PATH_HCE    = os.path.join(OUT_DIR, 'temperature_hce_v11.pt')
C2S_MODEL_NAME   = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# ── ORGAN_CONFIGS (training data; 6 organs, ids 0–5) ─────────────────────────
# v11: brain max_cells lowered 900→300 to match other organs (prevents brain-attractor dominance).
ORGAN_CONFIGS = [
    {'name': 'lung',         'path': 'lung.h5ad',                       'label_col': 'ann_finest_level', 'gene_col': 'feature_name', 'hierarchy_cols': ['ann_level_1','ann_level_2','ann_level_3','ann_level_4','ann_level_5'], 'coarse_col': None,                 'id': 0, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'brain_glia',   'path': 'brain_new.h5ad',                  'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': 'supercluster_term',  'id': 1, 'max_cells':  300, 'snrna_seq': True},
    {'name': 'brain_neurons','path': 'brain_neurons_processed.h5ad',    'label_col': 'label',            'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 2, 'max_cells':  300, 'snrna_seq': True},
    {'name': 'liver',        'path': 'census_data/liver.h5ad',          'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 3, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'lymph_node',   'path': 'census_data/lymph_node.h5ad',     'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 4, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'bone_marrow',  'path': 'census_data/bone_marrow.h5ad',    'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 5, 'max_cells':  500, 'snrna_seq': False},
]

# ── S2-C: roll back lung per-leaf caps ────────────────────────────────────────
PER_LEAF_MAX_BY_ORGAN = {}

# ── LAB_CONFIGS ───────────────────────────────────────────────────────────────
# v11: glioma and Brain_normal are snRNA-seq (median counts match training brain).
# Setting snrna_seq=True locks inference normalization to match training.
LAB_CONFIGS = [
    {'name': 'All_cells (glioma)', 'path': 'All_cells.h5ad',                  'label_col': 'predicted.high_hierarchy', 'snrna_seq': True},
    {'name': 'Brain_normal',       'path': 'lab-data/Brain_normal.h5ad',      'label_col': 'cell_type',                'snrna_seq': True},
    {'name': 'Liver_normal',       'path': 'lab-data/Liver_normal.h5ad',      'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'Lymph_node_normal',  'path': 'lab-data/Lymph_node_normal.h5ad', 'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'lymphoid',           'path': 'lab-data/lymphoid.h5ad',          'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'myeloid',            'path': 'lab-data/myeloid.h5ad',           'label_col': 'cell_type',                'snrna_seq': False},
]

TOP_K_GENES        = 200
MAX_CELLS_PER_TYPE = 300
MIN_CELLS_PER_TYPE = 100
TEST_FRAC          = 0.15
VAL_FRAC           = 0.10
BATCH_SIZE         = 16
N_EPOCHS           = 10
LEARNING_RATE      = 1e-4
WEIGHT_DECAY       = 1e-2
WARMUP_STEPS       = 200
MAX_SEQ_LEN        = 512
MAX_WEIGHT         = 15.0
SEED               = 42

# ── S2-B: targeted boosts resolved against post-dedup class_to_idx ────────────
TARGETED_WEIGHT_BOOSTS_BASE = [
    'CD4-positive, alpha-beta T cell',
    'regulatory T cell',
    'classical monocyte',
    'non-classical monocyte',
    'monocyte',
]

# Inference-time hyperparameters
TAU_LEAF   = 0.50

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device : {device}  |  Output dir : {OUT_DIR}')
print(f'  ORGAN_CONFIGS: {len(ORGAN_CONFIGS)} training organs')
print(f'  LAB_CONFIGS  : {len(LAB_CONFIGS)} zero-shot datasets')
print(f'  MAX_WEIGHT={MAX_WEIGHT}  |  boost bases={len(TARGETED_WEIGHT_BOOSTS_BASE)}')
print('[OK] Config ready (v11)')


  Device : cuda  |  Output dir : multi_tissue_v11_results
  ORGAN_CONFIGS: 6 training organs
  LAB_CONFIGS  : 6 zero-shot datasets
  MAX_WEIGHT=15.0  |  boost bases=5
[OK] Config ready (v11)


## 3. Data Loading & Preprocessing

In [3]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

def get_gene_symbols(adata):
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))

# ── Gene filter: remove snRNA-seq / housekeeping artifacts ───────────────────
EXCLUDE_PREFIXES = ('MT-', 'MTRNR', 'RPS', 'RPL', 'LINC', 'MIR',
                    'SNORD', 'SNORA', 'SNHG', 'ENSG', 'AC0', 'AC1',
                    'AL0', 'AL1', 'AP0', 'LOC')
EXCLUDE_EXACT    = {'MALAT1', 'NEAT1', 'XIST', 'TSIX', 'KCNQ1OT1',
                    'FTX', 'JPX', 'NORAD', 'HOTAIR', 'SOX2-OT',
                    'TUG1', 'GAS5', 'HOTAIRM1', 'PVT1'}
EXCLUDE_SUFFIXES = ('-AS1', '-AS2', '-AS3', '-AS4', '-AS5',
                    '-IT1', '-IT2', '-OT', '-OT1', '-OT2', '-OT3')

def is_keep_gene(g):
    g = str(g)
    if g in EXCLUDE_EXACT: return False
    if g.startswith(EXCLUDE_PREFIXES): return False
    if g.endswith(EXCLUDE_SUFFIXES): return False
    return True

def keep_gene_mask(gene_symbols):
    return np.array([is_keep_gene(g) for g in gene_symbols], dtype=bool)

def get_gene_lengths(adata, gene_symbols, default=2000.0):
    """Per-gene length from feature_length column (default fallback if missing)."""
    n = len(gene_symbols)
    if 'feature_length' in adata.var.columns:
        lengths = pd.to_numeric(adata.var['feature_length'], errors='coerce').values.astype(float)
        return np.where((lengths > 0) & np.isfinite(lengths), lengths, default)
    return np.full(n, default, dtype=float)


def load_organ(cfg, min_cells=MIN_CELLS_PER_TYPE, max_cells=None):
    if max_cells is None:
        max_cells = cfg.get('max_cells', MAX_CELLS_PER_TYPE)
    name, label_col = cfg['name'], cfg['label_col']
    t0 = time.time()
    adata = sc.read_h5ad(cfg['path'], backed='r')
    gene_symbols = get_gene_symbols(adata)
    valid_mask = adata.obs[label_col].apply(is_valid)
    obs        = adata.obs[valid_mask].copy()
    valid_idx  = np.where(valid_mask.values)[0]
    labels  = obs[label_col].astype(str).values
    # ── v8: per-leaf cap dict (e.g. lung fibroblast subtypes) overrides organ cap ─
    per_leaf_cap = PER_LEAF_MAX_BY_ORGAN.get(name, {})
    sampled = []
    capped_subtypes = []
    for lbl in np.unique(labels):
        pos = np.where(labels == lbl)[0]
        lbl_cap = per_leaf_cap.get(lbl, max_cells)
        if len(pos) > lbl_cap:
            pos = np.random.choice(pos, lbl_cap, replace=False)
            if lbl in per_leaf_cap:
                capped_subtypes.append((lbl, lbl_cap))
        sampled.extend(pos.tolist())
    if capped_subtypes:
        for lbl, cap in capped_subtypes:
            print(f'  [{name}] subtype-cap "{lbl}" → {cap}')
    sampled   = np.sort(np.array(sampled, dtype=np.int64))
    obs       = obs.iloc[sampled].copy()
    valid_idx = valid_idx[sampled]
    counts  = obs[label_col].value_counts()
    keep    = counts[counts >= min_cells].index
    dropped = counts[counts < min_cells]
    if len(dropped):
        print(f'  [{name}] Dropping {len(dropped)} type(s) < {min_cells} cells')
    mask      = obs[label_col].isin(keep)
    obs       = obs[mask].copy()
    valid_idx = valid_idx[mask.values]
    print(f'  [{name}] {len(obs):,} cells | {obs[label_col].nunique()} types | max_cells={max_cells} | snrna_seq={cfg.get("snrna_seq", False)} | {time.time()-t0:.1f}s')
    return adata, obs, valid_idx, gene_symbols


def cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k=200, desc='cells',
                        gene_lengths=None, length_normalize=False):
    gene_keep = keep_gene_mask(gene_symbols)
    if length_normalize and gene_lengths is None:
        raise ValueError('length_normalize=True requires gene_lengths')
    texts = []
    with h5py.File(h5_path, 'r') as f:
        X = f['X']
        indptr = X['indptr'][:]
        indices_ds, data_ds = X['indices'], X['data']
        for row_idx in tqdm(row_indices, desc=f'  {desc}', leave=False):
            start, end = int(indptr[row_idx]), int(indptr[row_idx + 1])
            if start == end: texts.append(''); continue
            vals = data_ds[start:end]
            cols = indices_ds[start:end]
            keep = gene_keep[cols]; vals = vals[keep]; cols = cols[keep]
            if len(vals) == 0: texts.append(''); continue
            score = vals / gene_lengths[cols] if length_normalize else vals
            if len(score) <= top_k:
                order = np.argsort(score)[::-1]
            else:
                order = np.argpartition(score, -top_k)[-top_k:]
                order = order[np.argsort(score[order])[::-1]]
            texts.append(' '.join(str(gene_symbols[cols[j]]) for j in order if vals[j] > 0))
    return texts


def cell_to_text_dense(X_row, gene_symbols, top_k=200, gene_keep=None,
                       gene_lengths=None, length_normalize=False):
    if gene_keep is None:
        gene_keep = keep_gene_mask(gene_symbols)
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    nz = nz[gene_keep[nz]]
    if len(nz) == 0: return ''
    vals  = row[nz]
    score = (vals / gene_lengths[nz]) if (length_normalize and gene_lengths is not None) else vals
    if len(nz) > top_k:
        idx = np.argpartition(score, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(score[idx])[::-1]]]
    else:
        nz = nz[np.argsort(score)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)


def cell_to_text_robust(h5_path, row_indices, gene_symbols, top_k=200, desc='cells',
                        gene_lengths=None, length_normalize=False):
    try:
        return cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k, desc,
                                    gene_lengths=gene_lengths, length_normalize=length_normalize)
    except (KeyError, AttributeError, TypeError, OSError):
        print(f'  [{desc}] CSR streaming failed — loading X into memory')
        adata_tmp = sc.read_h5ad(h5_path)
        X = adata_tmp.X
        gene_keep = keep_gene_mask(gene_symbols)
        texts = [cell_to_text_dense(X[i], gene_symbols, top_k, gene_keep=gene_keep,
                                     gene_lengths=gene_lengths, length_normalize=length_normalize)
                 for i in tqdm(row_indices, desc=f'  {desc}', leave=False)]
        del adata_tmp
        return texts

print('[OK] Helpers ready (with gene filter + length norm + per-organ + per-leaf caps)')

[OK] Helpers ready (with gene filter + length norm + per-organ + per-leaf caps)


In [4]:
print('Loading all organs ...')
loaded_organs = {}
for cfg in ORGAN_CONFIGS:
    adata, obs, valid_idx, gene_syms = load_organ(cfg)
    loaded_organs[cfg['name']] = {'cfg': cfg, 'adata': adata, 'obs': obs,
                                   'valid_idx': valid_idx, 'gene_symbols': gene_syms}

# ── Label normalization: disease filter (lymphoid/myeloid removed in v7) ─────
for organ_name in ['liver', 'lymph_node', 'bone_marrow']:
    d = loaded_organs[organ_name]
    obs = d['obs']
    if 'disease' in obs.columns:
        non_normal = (obs['disease'] != 'normal').sum()
        if non_normal:
            mask = obs['disease'] == 'normal'
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            print(f'  [{organ_name}] Removed {non_normal} non-normal cells')

# Fix lung pericyte naming
loaded_organs['lung']['obs']['ann_finest_level'] = (
    loaded_organs['lung']['obs']['ann_finest_level'].replace('Pericytes', 'pericyte')
)

print('[OK] All organs loaded and normalized')

Loading all organs ...
  [lung] Dropping 3 type(s) < 100 cells
  [lung] 17,700 cells | 59 types | max_cells=300 | snrna_seq=False | 27.6s
  [brain_glia] 3,900 cells | 13 types | max_cells=300 | snrna_seq=True | 2.9s
  [brain_neurons] 6,000 cells | 20 types | max_cells=300 | snrna_seq=True | 0.5s
  [liver] Dropping 793 type(s) < 100 cells
  [liver] 30,246 cells | 105 types | max_cells=300 | snrna_seq=False | 3.5s
  [lymph_node] Dropping 815 type(s) < 100 cells
  [lymph_node] 22,368 cells | 83 types | max_cells=300 | snrna_seq=False | 3.1s
  [bone_marrow] Dropping 789 type(s) < 100 cells
  [bone_marrow] 49,748 cells | 109 types | max_cells=500 | snrna_seq=False | 2.7s
[OK] All organs loaded and normalized


In [5]:
# v11: CANONICAL_LABEL_MAP imported from multi_tissue_v11_config.
# Applied to BOTH training organs and lab eval datasets so they share one canonical namespace.

# Bad label filters (organ-specific noise; unchanged from v10)
BAD_LABEL_FILTERS = [
    ('liver', 'cell_type', {'malignant cell'}),
    ('lymph_node', 'cell_type', {'stromal cell of pancreas', 'alveolar macrophage'}),
]
for organ_name, col, bad_labels in BAD_LABEL_FILTERS:
    d, obs = loaded_organs[organ_name], loaded_organs[organ_name]['obs']
    for lbl in bad_labels:
        n = (obs[col] == lbl).sum()
        if n:
            mask = obs[col] != lbl
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            obs = d['obs']
            print(f'  [{organ_name}] Dropped "{lbl}" ({n} cells)')

# Apply canonical map to training organs
total_remapped = 0
for cfg in ORGAN_CONFIGS:
    name, col = cfg['name'], cfg['label_col']
    obs = loaded_organs[name]['obs']
    for src, tgt in CANONICAL_LABEL_MAP.items():
        n = (obs[col] == src).sum()
        if n:
            loaded_organs[name]['obs'][col] = obs[col].replace(src, tgt)
            obs = loaded_organs[name]['obs']
            total_remapped += n
print(f'[OK] Canonical map applied to training organs — {total_remapped:,} cells remapped')


  [liver] Dropped "malignant cell" (300 cells)
  [lymph_node] Dropped "stromal cell of pancreas" (300 cells)
  [lymph_node] Dropped "alveolar macrophage" (114 cells)
[OK] Canonical map applied to training organs — 14,940 cells remapped


In [6]:
# ── Lung hierarchy edges (extracted from lung obs annotations) ───────────────
lung_cfg  = loaded_organs['lung']['cfg']
lung_obs2 = loaded_organs['lung']['obs']
lung_edges = {}
level_cols = [c for c in lung_cfg['hierarchy_cols'] if c in lung_obs2.columns]
cols_ordered = level_cols + [lung_cfg['label_col']]
for k in range(1, len(cols_ordered)):
    parent_col, child_col = cols_ordered[k-1], cols_ordered[k]
    for _, row in lung_obs2[[parent_col, child_col]].dropna().drop_duplicates().iterrows():
        p, ch = str(row[parent_col]), str(row[child_col])
        if is_valid_label(p) and is_valid_label(ch) and ch not in lung_edges:
            lung_edges[ch] = p
lung_roots = [str(v) for v in lung_obs2[cols_ordered[0]].dropna().unique()
              if is_valid_label(str(v))]

# ── Brain glia coarse→fine edges ─────────────────────────────────────────────
brain_glia_cfg = loaded_organs['brain_glia']['cfg']
brain_glia_obs = loaded_organs['brain_glia']['obs']
brain_glia_ontology = {}
coarse_col = brain_glia_cfg['coarse_col']
for _, row in brain_glia_obs[[coarse_col, brain_glia_cfg['label_col']]].dropna().drop_duplicates().iterrows():
    brain_glia_ontology[str(row[brain_glia_cfg['label_col']])] = str(row[coarse_col])
for val in brain_glia_obs[coarse_col].dropna().unique():
    if str(val) not in brain_glia_ontology:
        brain_glia_ontology[str(val)] = None

# ── Collect all canonical leaf labels from training organs ────────────────────
all_train_leaves = set()
for cfg in ORGAN_CONFIGS:
    obs = loaded_organs[cfg['name']]['obs']
    for lbl in obs[cfg['label_col']].dropna().unique():
        lbl = str(lbl)
        if is_valid_label(lbl):
            all_train_leaves.add(canonicalize(lbl))

# ── Build CL-backed combined ontology (v11) ────────────────────────────────────
# Prefer repo-root location of cl-basic.obo; fall back to relative path.
OBO_PATH = os.path.join(os.path.dirname(os.path.abspath('.')), 'ontology', 'cl-basic.obo')
if not os.path.exists(OBO_PATH):
    OBO_PATH = 'ontology/cl-basic.obo'
combined_ontology, unresolved = build_combined_ontology(
    canonical_leaves=all_train_leaves,
    lung_edges=lung_edges,
    lung_roots=lung_roots,
    obo_path=OBO_PATH,
    strict=True,
)

# Graft brain glia coarse→fine without overriding CL-derived edges
for child, parent in brain_glia_ontology.items():
    combined_ontology.setdefault(child, parent)

# Fallback: add any unresolved leaves as orphan roots so they appear in the ontology
for lbl in unresolved:
    combined_ontology.setdefault(lbl, None)
if unresolved:
    print(f'  WARNING: {len(unresolved)} unresolved leaves (added as orphan roots): {unresolved[:10]}')

# Deduplicate collisions (leaf label that is also an interior node)
for cfg in ORGAN_CONFIGS:
    obs, combined_ontology, report = deduplicate_hierarchy(
        loaded_organs[cfg['name']]['obs'], combined_ontology, cfg['label_col']
    )
    loaded_organs[cfg['name']]['obs'] = obs
    if not report.empty:
        for _, row in report.iterrows():
            print(f'  [{cfg["name"]}] "{row["original_label"]}" -> "{row["new_label"]}" ({row["n_cells_renamed"]})')

# Ensure canonical-map targets are reachable in the ontology
for tgt in set(CANONICAL_LABEL_MAP.values()):
    if is_valid_label(tgt) and tgt not in combined_ontology:
        combined_ontology[tgt] = None

print(f'[OK] Combined ontology (v11 CL-backed): {len(combined_ontology)} entries')
if unresolved:
    print(f'  Unresolved: {unresolved}')


  [lung] "AT2" -> "AT2 (unspecified)" (300)
  [lung] "B cell" -> "B cell (unspecified)" (300)
  [lung] "CD4-positive, alpha-beta T cell" -> "CD4-positive, alpha-beta T cell (unspecified)" (300)
  [lung] "CD8-positive, alpha-beta T cell" -> "CD8-positive, alpha-beta T cell (unspecified)" (300)
  [lung] "natural killer cell" -> "natural killer cell (unspecified)" (300)
  [lung] "plasma cell" -> "plasma cell (unspecified)" (300)
  [lung] "smooth muscle cell" -> "smooth muscle cell (unspecified)" (300)
  [brain_glia] "astrocyte" -> "astrocyte (unspecified)" (300)
  [brain_glia] "endothelial cell" -> "endothelial cell (unspecified)" (300)
  [brain_glia] "ependymal cell" -> "ependymal cell (unspecified)" (300)
  [brain_glia] "fibroblast" -> "fibroblast (unspecified)" (300)
  [brain_neurons] "medium spiny neuron" -> "medium spiny neuron (unspecified)" (300)
  [liver] "B cell" -> "B cell (unspecified)" (600)
  [liver] "B-2 B cell" -> "B-2 B cell (unspecified)" (300)
  [liver] "CD4-positive, al

In [7]:
print('Cell-to-text conversion ...')
CSR_ORGANS = {'lung', 'brain_glia', 'brain_neurons', 'liver', 'lymph_node', 'bone_marrow'}
all_texts, all_labels_str, all_organ_ids = [], [], []
organ_texts, organ_labels = {}, {}

for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    d    = loaded_organs[name]
    gene_lengths = get_gene_lengths(d['adata'], d['gene_symbols'])
    length_norm  = cfg.get('snrna_seq', False)
    print(f'  [{name}] {len(d["valid_idx"]):,} cells | length_normalize={length_norm} ...')
    fn = cell_to_text_backed if name in CSR_ORGANS else cell_to_text_robust
    texts = fn(cfg['path'], d['valid_idx'], d['gene_symbols'], TOP_K_GENES,
               desc=name, gene_lengths=gene_lengths, length_normalize=length_norm)
    keep_mask = [bool(t.strip()) for t in texts]
    texts     = [t for t, k in zip(texts, keep_mask) if k]
    labels    = d['obs'][cfg['label_col']].astype(str).values[keep_mask]
    organ_texts[name], organ_labels[name] = texts, labels
    all_texts.extend(texts)
    all_labels_str.extend(labels)
    all_organ_ids.extend([cfg['id']] * len(texts))

all_labels_str = np.array(all_labels_str)
all_organ_ids  = np.array(all_organ_ids, dtype=int)
print(f'[OK] {len(all_texts):,} cells | {len(np.unique(all_labels_str))} unique labels')

Cell-to-text conversion ...
  [lung] 17,700 cells | length_normalize=False ...


  [brain_glia] 3,900 cells | length_normalize=True ...


  [brain_neurons] 6,000 cells | length_normalize=True ...


  [liver] 29,946 cells | length_normalize=False ...


  [lymph_node] 21,954 cells | length_normalize=False ...


  [bone_marrow] 49,748 cells | length_normalize=False ...


[OK] 129,248 cells | 225 unique labels


In [8]:
print('Building vocabulary and splits ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

leaf_classes_set = set(all_labels_str)
all_nodes = set()
for child, parent in combined_ontology.items():
    all_nodes.add(child)
    if parent: all_nodes.add(parent)
all_nodes |= leaf_classes_set

class_names   = sorted(all_nodes)
class_to_idx  = {name: idx for idx, name in enumerate(class_names)}
n_classes     = len(class_names)
leaf_classes  = sorted(leaf_classes_set)
leaf_indices  = [class_to_idx[c] for c in leaf_classes]
leaf_index_set = set(leaf_indices)

# Leaf-local index mapping (for CE baseline)
leaf_to_local   = {leaf_idx: i for i, leaf_idx in enumerate(leaf_indices)}
local_to_leaf   = {i: leaf_idx for i, leaf_idx in enumerate(leaf_indices)}
n_leaf_classes  = len(leaf_classes)

labels_encoded = np.array([class_to_idx[s] for s in all_labels_str], dtype=int)

class CellTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

strat_key = labels_encoded * 10 + all_organ_ids
idx_all   = np.arange(len(all_texts))
idx_tv, idx_test   = train_test_split(idx_all, test_size=TEST_FRAC, stratify=strat_key, random_state=SEED)
idx_train, idx_val = train_test_split(idx_tv,  test_size=VAL_FRAC/(1-TEST_FRAC), stratify=strat_key[idx_tv], random_state=SEED)

train_labels = labels_encoded[idx_train]
val_labels   = labels_encoded[idx_val]
test_labels  = labels_encoded[idx_test]
test_organ   = all_organ_ids[idx_test]

train_ds = CellTextDataset([all_texts[i] for i in idx_train], train_labels, tokenizer, MAX_SEQ_LEN)
val_ds   = CellTextDataset([all_texts[i] for i in idx_val],   val_labels,   tokenizer, MAX_SEQ_LEN)
test_ds  = CellTextDataset([all_texts[i] for i in idx_test],  test_labels,  tokenizer, MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'  Total vocab : {n_classes}  |  Leaf classes : {n_leaf_classes}')
print(f'  Train : {len(idx_train):,} | Val : {len(idx_val):,} | Test : {len(idx_test):,}')
print('[OK] Vocabulary and splits ready')

Building vocabulary and splits ...
  Total vocab : 386  |  Leaf classes : 225
  Train : 96,935 | Val : 12,925 | Test : 19,388
[OK] Vocabulary and splits ready


## 3.5 Diagnostic: training-set class counts for lineage-critical labels


In [9]:
# v8 diagnostic: surfaces actual CD4/CD8/monocyte/macrophage counts AFTER all
# label normalization + caps + min-cells filter. Use this to decide if the
# targeted weight boosts in Cell 4 are sufficient (or if data needs rebalancing).
from collections import Counter
counts = Counter(class_names[i] for i in train_labels)

KEYS_OF_INTEREST = [
    'CD4-positive, alpha-beta T cell',
    'CD8-positive, alpha-beta T cell',
    'regulatory T cell',
    'classical monocyte',
    'non-classical monocyte',
    'monocyte',
    'macrophage',
    'macrophage (unspecified)',
    'alveolar macrophage',
    'plasma cell',
    'B cell',
    'natural killer cell',
    'neuron (unspecified)',
    'oligodendrocyte',
    'astrocyte',
]
print('  v8 training counts for lineage-critical classes:')
print(f'  {"Class":<48} {"count":>8}')
print('  ' + '-' * 58)
for k in KEYS_OF_INTEREST:
    print(f'  {k:<48} {counts.get(k, 0):>8}')

cd4 = counts.get('CD4-positive, alpha-beta T cell', 0)
cd8 = counts.get('CD8-positive, alpha-beta T cell', 0)
if cd4 and cd8:
    ratio = cd8 / max(cd4, 1)
    if ratio >= 2.0:
        print(f'\n  ⚠ CD8/CD4 ratio = {ratio:.2f}× — TARGETED_WEIGHT_BOOSTS in Cell 4 should help.')
        print(f'    If still biased after retrain, downsample CD8 in load step or boost CD4 further.')
    else:
        print(f'\n  CD8/CD4 ratio = {ratio:.2f}× — balanced.')

mono = counts.get('classical monocyte', 0) + counts.get('non-classical monocyte', 0) + counts.get('monocyte', 0)
mac  = counts.get('macrophage', 0) + counts.get('macrophage (unspecified)', 0) + counts.get('alveolar macrophage', 0)
if mono and mac:
    print(f'  monocyte total ≈ {mono}   macrophage total ≈ {mac}   ratio mac/mono = {mac/max(mono,1):.2f}')
    if mono < 200:
        print(f'  ⚠ Monocyte training cells < 200 — predictions may default to macrophage.')


  v8 training counts for lineage-critical classes:
  Class                                               count
  ----------------------------------------------------------
  CD4-positive, alpha-beta T cell                         0
  CD8-positive, alpha-beta T cell                         0
  regulatory T cell                                    1400
  classical monocyte                                   1425
  non-classical monocyte                               1049
  monocyte                                                0
  macrophage                                              0
  macrophage (unspecified)                             1050
  alveolar macrophage                                   225
  plasma cell                                             0
  B cell                                                  0
  natural killer cell                                     0
  neuron (unspecified)                                  142
  oligodendrocyte                               

## 4. Loss Function & Class Weights *(HCE only)*

In [10]:
print('Building reachability matrix and class weights ...')

# ── Reachability matrix (HCE only) ──────────────────────────────────────────
R_np = build_reachability_matrix_from_ontology(combined_ontology, class_names)
reachability_matrix = torch.tensor(R_np, dtype=torch.float32).to(device)
print(f'  R shape : {n_classes}x{n_classes} | nnz={int(reachability_matrix.sum())}')

# ── HCE class weights (full vocab, ancestor effective-count) ────────────────
train_counts   = Counter(train_labels.tolist())
N_train, C_obs = len(train_labels), len(train_counts)

class_weights_hce = torch.zeros(n_classes, dtype=torch.float32, device=device)
for idx_ct, count in train_counts.items():
    class_weights_hce[idx_ct] = N_train / (C_obs * count)
ancestor_indices = [i for i in range(n_classes) if i not in leaf_index_set]
for anc_idx in ancestor_indices:
    eff = sum(train_counts.get(j, 0) for j in leaf_indices if R_np[anc_idx, j] > 0)
    if eff > 0:
        class_weights_hce[anc_idx] = N_train / (C_obs * eff)
class_weights_hce = torch.clamp(class_weights_hce, max=MAX_WEIGHT)

# ── S2-A: down-weight (unspecified) leaves 0.2x so they don't dominate ──────
unspec_names = [cn for cn in class_names if cn.endswith(' (unspecified)')]
for cls_name in unspec_names:
    class_weights_hce[class_to_idx[cls_name]] *= 0.2
print(f'  S2-A: down-weighted {len(unspec_names)} "(unspecified)" leaves by 0.2x')

# ── S2-B: resolve boost names against post-dedup class_to_idx ───────────────
print('  S2-B: resolving TARGETED_WEIGHT_BOOSTS_BASE post-dedup:')
for base in TARGETED_WEIGHT_BOOSTS_BASE:
    if base in class_to_idx:
        class_weights_hce[class_to_idx[base]] *= 1.5
        print(f'    boost 1.5x → leaf "{base}"')
    elif f'{base} (unspecified)' in class_to_idx:
        # base was renamed by dedup; boost specific children instead (not the unspec leaf)
        children = [n for n,p in combined_ontology.items()
                    if p == base and not n.endswith(' (unspecified)') and n in class_to_idx]
        if children:
            for ch in children:
                class_weights_hce[class_to_idx[ch]] *= 1.3
                print(f'    boost 1.3x → child "{ch}" (parent "{base}" was deduped)')
        else:
            print(f'    skip: "{base}" → "(unspecified)" with no specific children to boost')
    else:
        print(f'    skip: "{base}" not in vocab')


# ── S2-E: attractor down-weighting (v11) ────────────────────────────────────
# v10 analysis identified these leaves as absorbing unrelated probability mass.
# The OPC merge (CANONICAL_LABEL_MAP) already fixes committed-OPC; the others
# are pre-emptively down-weighted to spread distribution more specifically.
KNOWN_ATTRACTOR_LEAVES = [
    'Interstitial Mph perivascular',   # captured 30-50% of many myeloid true classes
    'Peribronchial fibroblasts',        # captured >40% of stromal true classes
    'natural killer cell',              # captured >25% of lymphoid classes
]
n_attractor_down = 0
for lbl in KNOWN_ATTRACTOR_LEAVES:
    canonical_lbl = canonicalize(lbl)
    target = canonical_lbl if canonical_lbl in class_to_idx else (lbl if lbl in class_to_idx else None)
    if target is not None:
        class_weights_hce[class_to_idx[target]] *= 0.5
        print(f'  S2-E: attractor down-weight 0.5x → "{target}"')
        n_attractor_down += 1
print(f'  S2-E: {n_attractor_down}/{len(KNOWN_ATTRACTOR_LEAVES)} attractor leaves down-weighted')

hce_criterion = HCELoss(reachability_matrix, class_weights_hce).to(device)
print('\n[OK] HCE loss ready')
print(f'  HCE class weights : {(class_weights_hce > 0).sum().item()}/{n_classes} non-zero')


Building reachability matrix and class weights ...
  R shape : 386x386 | nnz=2273
  S2-A: down-weighted 48 "(unspecified)" leaves by 0.2x
  S2-B: resolving TARGETED_WEIGHT_BOOSTS_BASE post-dedup:
    boost 1.5x → leaf "CD4-positive, alpha-beta T cell"
    boost 1.5x → leaf "regulatory T cell"
    boost 1.5x → leaf "classical monocyte"
    boost 1.5x → leaf "non-classical monocyte"
    boost 1.5x → leaf "monocyte"
  S2-E: attractor down-weight 0.5x → "Interstitial Mph perivascular"
  S2-E: attractor down-weight 0.5x → "Peribronchial fibroblasts"
  S2-E: attractor down-weight 0.5x → "natural killer cell"
  S2-E: 3/3 attractor leaves down-weighted


NameError: name 'HCELoss' is not defined

## 4.5 S1-D Diagnostic — which `TARGETED_WEIGHT_BOOSTS` actually applied (post-dedup)


In [ ]:
# v10 S1-D: verify Stage 2's boost-key resolution.
# Cell 15 already prints "boost 1.5x → leaf" or "boost 1.3x → child" lines per base.
# This cell just summarizes for a quick sanity glance.
print('  v10 boost-base resolution summary:')
print(f'  {"base":<46}  {"resolution"}')
print('  ' + '-' * 90)
for base in TARGETED_WEIGHT_BOOSTS_BASE:
    if base in class_to_idx:
        kind = 'LEAF (1.5x applied)'
    elif f'{base} (unspecified)' in class_to_idx:
        children = [n for n,p in combined_ontology.items()
                    if p == base and not n.endswith(' (unspecified)') and n in class_to_idx]
        kind = f'DEDUPED → {len(children)} child(ren) got 1.3x'
    else:
        kind = 'NOT IN VOCAB'
    print(f'  {base:<46}  {kind}')


## 5. Model Architecture

In [ ]:
class C2SClassifier(nn.Module):
    """C2S-Pythia-410m encoder + dropout + linear head over n_classes."""
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(hidden_size, num_classes)
    def forward(self, input_ids, attention_mask):
        out         = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        seq_len     = attention_mask.sum(dim=1) - 1
        last_token  = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))

def build_fresh_model():
    """Load a fresh C2S encoder and wrap in C2SClassifier."""
    enc = AutoModel.from_pretrained(C2S_MODEL_NAME)
    enc.gradient_checkpointing_enable()
    return C2SClassifier(enc, enc.config.hidden_size, n_classes).to(device)

def build_optimizer_and_schedulers(model, n_steps):
    opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_STEPS)
    cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_steps - WARMUP_STEPS))
    return opt, warmup, cosine

total_params = sum(p.numel() for p in build_fresh_model().parameters()) / 1e6
print(f'[OK] Architecture: C2S-Pythia-410m + head  |  {total_params:.1f}M params  |  {n_classes} output classes')

## 6. Training Loop

In [ ]:
leaf_indices_t = torch.tensor(leaf_indices, device=device)

def train_model(model, criterion, save_path, label):
    """
    Train model for N_EPOCHS, save best checkpoint by val accuracy.
    Returns history dict and best_val_acc.
    """
    total_steps = len(train_loader) * N_EPOCHS
    optimizer, warmup_sched, cosine_sched = build_optimizer_and_schedulers(model, total_steps)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, global_step = 0.0, 0

    print(f'\n{"="*60}')
    print(f'  Training [{label}]')
    print(f'  Steps/epoch: {len(train_loader)}  |  Total: {total_steps}')
    print('=' * 60)
    t_start = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        t_epoch = time.time()
        model.train()
        running_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [train]', leave=True)
        for batch in pbar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['label'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if global_step < WARMUP_STEPS: warmup_sched.step()
            else: cosine_sched.step()
            global_step += 1
            running_loss += loss.item()
            n_batches    += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        train_loss = running_loss / n_batches

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [val]  ', leave=False):
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels_b       = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)
                val_loss_sum  += criterion(logits, labels_b).item() * len(labels_b)
                preds          = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
                val_correct   += (preds == labels_b).sum().item()
                val_total     += len(labels_b)

        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_acc': val_acc, 'label': label}, save_path)

        print(f'  [{label}] Epoch {epoch}/{N_EPOCHS} | train={train_loss:.4f} | '
              f'val={val_loss:.4f} | val_acc={val_acc:.4f} | {"** BEST **" if improved else ""} | {time.time()-t_epoch:.0f}s')

    print(f'\n  [{label}] Total time : {(time.time()-t_start)/60:.1f} min  |  Best val acc : {best_val_acc:.4f}')
    return history, best_val_acc

print('[OK] Training loop defined')

## 7. Train HCE Model

In [ ]:
# v11: train HCE model only.
hce_model = build_fresh_model()
if os.path.exists(HCE_MODEL_PATH):
    print(f'  Loading existing HCE checkpoint: {HCE_MODEL_PATH}')
    ckpt = torch.load(HCE_MODEL_PATH, map_location=device)
    sd = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    miss, unex = hce_model.load_state_dict(sd, strict=False)
    if miss or unex:
        print(f'  WARNING: HCE state_dict mismatch: missing={len(miss)} unexpected={len(unex)}')
    hce_history = None
    hce_best_acc = ckpt.get('best_val_acc') if isinstance(ckpt, dict) else None
    print('  [OK] HCE v11 checkpoint loaded — skipping training')
else:
    print(f'  Training HCE v11 ({N_EPOCHS} epochs)')
    hce_history, hce_best_acc = train_model(hce_model, hce_criterion, HCE_MODEL_PATH, 'HCE')


## 7.5 Temperature Calibration (fit on validation)


In [ ]:
# v11: fit (or load) temperature for HCE model only.

def _collect_val_logits(model, leaf_indices_t):
    model.eval()
    logits_all, labels_all = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  val logits', leave=False):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            logits_all.append(logits[:, leaf_indices_t].detach().cpu())
            labels_all.append(batch['label'].cpu())
    return torch.cat(logits_all, dim=0), torch.cat(labels_all, dim=0)

def _fit_temperature(model, model_path, label, save_path):
    if os.path.exists(save_path):
        T_val = float(torch.load(save_path, map_location='cpu'))
        print(f'  [{label}] Loaded existing temperature T={T_val:.3f} from {save_path}')
        return T_val
    print(f'  [{label}] Fitting temperature on validation set ...')
    ckpt = torch.load(model_path, map_location=device)
    sd = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(sd, strict=False)
    val_logits, val_labels = _collect_val_logits(model, leaf_indices_t)
    val_local = torch.tensor([leaf_to_local.get(int(l), -1) for l in val_labels.tolist()], dtype=torch.long)
    valid = val_local >= 0
    val_logits_f = val_logits[valid].to(device)
    val_local_f  = val_local[valid].to(device)
    print(f'    Val cells with leaf labels: {valid.sum().item():,}/{len(val_local):,}')
    T = torch.tensor([1.0], device=device, requires_grad=True)
    nll = nn.CrossEntropyLoss()
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nll(val_logits_f / T.clamp(min=0.05), val_local_f)
        loss.backward()
        return loss
    opt.step(closure)
    T_val = float(T.clamp(min=0.05).item())

    def _ece(probs, lbls, n_bins=15):
        confs, preds = probs.max(dim=1)
        accs = (preds == lbls).float()
        bins = torch.linspace(0, 1, n_bins+1)
        ece = 0.0
        for k in range(n_bins):
            m = (confs > bins[k]) & (confs <= bins[k+1])
            if m.sum() > 0:
                ece += (m.float().mean() * (accs[m].mean() - confs[m].mean()).abs()).item()
        return ece
    p_pre  = torch.softmax(val_logits_f, dim=1).cpu()
    p_post = torch.softmax(val_logits_f / T_val, dim=1).cpu()
    print(f'    [{label}] Fitted T={T_val:.3f}   ECE: {_ece(p_pre, val_local_f.cpu())*100:.2f}% → {_ece(p_post, val_local_f.cpu())*100:.2f}%')
    torch.save(T_val, save_path)
    print(f'    [OK] Saved to {save_path}')
    return T_val

TEMPERATURE_HCE = _fit_temperature(hce_model, HCE_MODEL_PATH, 'HCE', TEMP_PATH_HCE)


## 8. Evaluate HCE on Test Set

In [ ]:
def evaluate_model(model, ckpt_path, label):
    ckpt = torch.load(ckpt_path, map_location=device)
    sd = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(sd, strict=False)
    if isinstance(ckpt, dict) and 'epoch' in ckpt and 'val_acc' in ckpt:
        print(f'  [{label}] Loaded epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f}')
    else:
        print(f'  [{label}] Loaded checkpoint from {ckpt_path}')
    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'  [{label}] Evaluating', leave=True):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds  = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
            preds_all.extend(preds.cpu().numpy())
            trues_all.extend(batch['label'].numpy())
    return np.array(preds_all), np.array(trues_all)


def compute_metrics(trues, preds, organ_id=None):
    if organ_id is not None:
        mask  = test_organ == organ_id
        trues = trues[mask]; preds = preds[mask]
    if len(trues) == 0: return None
    unique = np.unique(trues)
    acc    = accuracy_score(trues, preds)
    p, r, f, _ = precision_recall_fscore_support(trues, preds, labels=unique, average='macro', zero_division=0)
    p_pc, r_pc, f_pc, sup = precision_recall_fscore_support(trues, preds, labels=unique, zero_division=0)
    per_class = pd.DataFrame({
        'cell_type': [class_names[i] for i in unique],
        'precision': p_pc, 'recall': r_pc, 'f1': f_pc, 'support': sup,
    })
    return {
        'accuracy': acc, 'macro_recall': r, 'macro_f1': f, 'macro_precision': p,
        'zero_recall_count': int((per_class['recall'] == 0).sum()),
        'recall_50_pct': float((per_class['recall'] >= 0.5).mean()),
        'recall_80_pct': float((per_class['recall'] >= 0.8).mean()),
        'per_class': per_class,
        'n_samples': len(trues),
    }

print('[OK] evaluate_model + compute_metrics defined (HCE only)')


## 9. HCE Test Set Metrics

In [ ]:
print('=' * 80)
print('  HCE v11 — TEST SET METRICS')
print('=' * 80)

print('\nEvaluating HCE v11 ...')
hce_preds, hce_trues = evaluate_model(hce_model, HCE_MODEL_PATH, 'HCE')

# Per-organ + COMBINED
results_hce = {}
for cfg in ORGAN_CONFIGS:
    m = compute_metrics(hce_trues, hce_preds, cfg['id'])
    if m: results_hce[cfg['name']] = m
results_hce['COMBINED'] = compute_metrics(hce_trues, hce_preds)

metric_cols = ['accuracy', 'macro_recall', 'macro_f1', 'zero_recall_count', 'recall_50_pct', 'recall_80_pct']
def _fmt(v, col):
    return f'{v:.0f}' if col == 'zero_recall_count' else f'{v:.4f}'

print(f'\n  {"organ":<14} ' + ''.join(f'{c:>16}' for c in metric_cols))
for organ_name, m in results_hce.items():
    print(f'  {organ_name:<14} ' + ''.join(f'{_fmt(m[c], c):>16}' for c in metric_cols))

# Save metrics CSV
def _to_df(res):
    rows = []
    for organ_name, m in res.items():
        rows.append({'organ': organ_name, **{c: m[c] for c in metric_cols}, 'n_samples': m['n_samples']})
    return pd.DataFrame(rows)
_to_df(results_hce).to_csv(os.path.join(OUT_DIR, 'hce_v11_test_metrics.csv'), index=False)
print(f'\n[OK] Saved metrics CSV to {OUT_DIR}/')


## 10. Training Curves

In [ ]:
# v9 Stage 1: skip training curves when reusing v8 checkpoint (no history available)
if hce_history is None:
    print('  Skipping training curves — checkpoint loaded from v8 (no training history)')
else:
    epochs_ax = np.arange(1, N_EPOCHS + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs_ax, hce_history['val_acc'], 'o-', label='HCE v9')
    axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Acc')
    axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs_ax, hce_history['val_loss'], 'o-', label='HCE v9')
    axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(epochs_ax, hce_history['train_loss'], 'o-', label='HCE v9')
    axes[2].set_title('Training Loss'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Train Loss')
    axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle('HCE v9 Training Curves')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'hce_v9_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 11. Per-Class Recall by Organ

In [ ]:
# v11: per-class recall — HCE only
def plot_recall_hce(organ_name, hce_metrics):
    hce_pc = hce_metrics['per_class'].set_index('cell_type')['recall'].sort_values()
    all_ct = hce_pc.index.tolist()
    n = len(all_ct)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.30)))
    y = np.arange(n)
    ax.barh(y, hce_pc.values, color='steelblue', alpha=0.85)
    ax.axvline(0.5, color='gray', ls='--', lw=1, label='50%')
    ax.axvline(0.8, color='gray', ls=':',  lw=1, label='80%')
    ax.set_yticks(y); ax.set_yticklabels(all_ct, fontsize=7)
    ax.set_xlim(0, 1.05); ax.set_xlabel('Recall')
    ax.set_title(f'Per-class Recall — {organ_name}  (macro={hce_metrics["macro_recall"]:.3f})')
    ax.legend(loc='lower right', fontsize=9); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    safe = organ_name.replace(' ', '_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(os.path.join(OUT_DIR, f'recall_hce_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.show()

for cfg in ORGAN_CONFIGS:
    if cfg['name'] in results_hce:
        plot_recall_hce(cfg['name'], results_hce[cfg['name']])
plot_recall_hce('COMBINED', results_hce['COMBINED'])


## 12. Recall Distribution

In [ ]:
hce_recalls = results_hce['COMBINED']['per_class']['recall'].values
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(hce_recalls, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.6, label='50% threshold')
axes[0].set_title(f'Recall Distribution (n={len(hce_recalls)} classes)')
axes[0].set_xlabel('Per-class recall'); axes[0].set_ylabel('# classes')
axes[0].legend(); axes[0].grid(alpha=0.3)

sorted_r = np.sort(hce_recalls)
cdf      = np.arange(1, len(sorted_r) + 1) / len(sorted_r)
axes[1].plot(sorted_r, cdf, '-', linewidth=2, color='steelblue')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('CDF of per-class recall'); axes[1].set_xlabel('Recall threshold'); axes[1].set_ylabel('Fraction of classes')
axes[1].grid(alpha=0.3)

print(f'  HCE — mean recall: {hce_recalls.mean():.4f}  |  median: {np.median(hce_recalls):.4f}  |  0% classes: {(hce_recalls == 0).sum()}')

plt.suptitle('HCE v11 — Combined Test-Set Recall')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v11_recall_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. Zero-Shot Inference *(6 lab datasets)*

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] if self.texts[idx].strip() else '[PAD]',
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}


# ── S1-A: mask "(unspecified)" leaves from leaf argmax ──────────────────────
specific_leaf_classes = [c for c in leaf_classes if not c.endswith(' (unspecified)')]
specific_leaf_indices = [class_to_idx[c] for c in specific_leaf_classes]
specific_leaf_indices_t = torch.tensor(specific_leaf_indices, device=device)
masked_count = len(leaf_indices) - len(specific_leaf_indices)
print(f'  S1-A: masked {masked_count} "(unspecified)" leaves  ({len(specific_leaf_indices)} specific leaves remain)')


# ── Ancestor reachability + node depths (for HCE roll-up) ───────────────────
ancestors_of_leaf = {}
for leaf_idx in leaf_indices:
    ancestors_of_leaf[leaf_idx] = [i for i in range(n_classes) if R_np[i, leaf_idx] > 0]

def _compute_depths():
    depths = {name: None for name in class_names}
    def depth_of(name, seen=None):
        seen = seen or set()
        if name in seen: return 0
        if depths[name] is not None: return depths[name]
        parent = combined_ontology.get(name)
        if parent is None or parent not in depths:
            depths[name] = 0
        else:
            depths[name] = 1 + depth_of(parent, seen | {name})
        return depths[name]
    for nm in class_names: depth_of(nm)
    return depths
node_depth = _compute_depths()
depth_arr  = np.array([node_depth[nm] for nm in class_names], dtype=int)


# ── S1-C: adaptive roll-up (HCE only) ───────────────────────────────────────
def rollup_pred(leaf_probs_row, tau_leaf=TAU_LEAF):
    leaf_max_idx = int(np.argmax(leaf_probs_row))
    leaf_max_p   = float(leaf_probs_row[leaf_max_idx])
    if leaf_max_p >= tau_leaf:
        return leaf_indices[leaf_max_idx], leaf_max_p, False
    s = np.zeros(n_classes, dtype=float)
    for k, leaf_idx in enumerate(leaf_indices):
        p = leaf_probs_row[k]
        if p <= 0: continue
        for anc_idx in ancestors_of_leaf[leaf_idx]:
            s[anc_idx] += p
    tau_adaptive = max(0.4, min(1.0, 2.5 * leaf_max_p))
    candidate_mask = s >= tau_adaptive
    if not candidate_mask.any():
        return leaf_indices[leaf_max_idx], leaf_max_p, False
    cand_idx = np.where(candidate_mask)[0]
    best = max(cand_idx, key=lambda i: (depth_arr[i], s[i]))
    return int(best), float(s[best]), True


def run_inference_hce(model, texts, temperature=1.0):
    """HCE: S1-A specific-leaf argmax + S1-C adaptive rollup on full leaf set."""
    inf_loader = DataLoader(
        InferenceDataset(texts, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True
    )
    leaf_preds, leaf_confs = [], []
    lin_preds, lin_confs, lin_rolled = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(inf_loader, desc='    [HCE] infer', leave=False):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            spec = torch.softmax(logits[:, specific_leaf_indices_t] / temperature, dim=1)
            pp = spec.argmax(dim=1)
            leaf_preds.extend(specific_leaf_indices_t[pp].cpu().numpy())
            leaf_confs.extend(spec.max(dim=1).values.cpu().numpy())
            full = torch.softmax(logits[:, leaf_indices_t] / temperature, dim=1).cpu().numpy()
            for row in full:
                idx, p, rolled = rollup_pred(row)
                lin_preds.append(idx); lin_confs.append(p); lin_rolled.append(rolled)
    return ([class_names[i] for i in leaf_preds], np.array(leaf_confs),
            [class_names[i] for i in lin_preds],  np.array(lin_confs), np.array(lin_rolled))


lab_results_hce = {}

for lab_cfg in LAB_CONFIGS:
    lab_name  = lab_cfg['name']
    label_col = lab_cfg['label_col']
    sn_flag   = lab_cfg.get('snrna_seq', False)
    print(f'\n  --- {lab_name} ---  (snrna_seq={sn_flag})')

    adata_lab = sc.read_h5ad(lab_cfg['path'])
    lab_gene_sym = get_gene_symbols(adata_lab)
    lab_gene_keep = keep_gene_mask(lab_gene_sym)
    lab_gene_lengths = get_gene_lengths(adata_lab, lab_gene_sym) if sn_flag else None
    true_labels = adata_lab.obs[label_col].astype(str).values
    # v11: canonicalize lab labels to the shared namespace
    true_labels = np.array([canonicalize(l) for l in true_labels])
    X_lab = adata_lab.X
    texts = [cell_to_text_dense(X_lab[i], lab_gene_sym, TOP_K_GENES,
                                 gene_keep=lab_gene_keep,
                                 gene_lengths=lab_gene_lengths,
                                 length_normalize=sn_flag)
             for i in tqdm(range(adata_lab.n_obs), desc='    text', leave=False)]

    h_leaf, h_conf, h_lin, h_lconf, h_rolled = run_inference_hce(hce_model, texts, TEMPERATURE_HCE)
    lab_results_hce[lab_name] = pd.DataFrame({
        'true_label': true_labels,
        'pred_leaf': h_leaf, 'conf_leaf': h_conf,
        'pred_lineage': h_lin, 'conf_lineage': h_lconf, 'rolled_up': h_rolled,
    })

    print(f'  HCE roll-up rate: {h_rolled.mean()*100:.1f}%')
    df_h = lab_results_hce[lab_name]
    for true_lbl in sorted(np.unique(true_labels)):
        sub_h = df_h[df_h['true_label']==true_lbl]
        ht = sub_h['pred_leaf'].value_counts()
        nt = sub_h['pred_lineage'].value_counts()
        def _top(vc):
            return f'{vc.index[0]}  ({vc.iloc[0]/vc.sum()*100:.0f}%)' if len(vc) else 'N/A'
        print(f'  {true_lbl:<40} {_top(ht):>40} {_top(nt):>40}')

print('\n[OK] Zero-shot inference complete (HCE v11)')


## 14. Deeper Biological-Accuracy Analysis

Top-5 predictions per true class, lineage-match rate, and ancestor recall on the in-distribution test set.

In [ ]:
# v11: LINEAGE_RULES and assign_lineage imported from multi_tissue_v11_config.
# Glioma (All_cells) is reported lineage-only; excluded from leaf-accuracy rows.

GLIOMA_DATASET = 'All_cells (glioma)'

# (1) Top-5 predictions per true class (leaf accuracy; skip glioma)
print('=' * 100)
print('  (1) TOP-5 PREDICTIONS PER TRUE CLASS (zero-shot; glioma excluded — lineage-only)')
print('=' * 100)
for lab_name in lab_results_hce.keys():
    if lab_name == GLIOMA_DATASET:
        continue
    print(f'\n--- {lab_name} ---')
    df = lab_results_hce[lab_name]
    for true_lbl in sorted(df['true_label'].unique()):
        n = int((df['true_label'] == true_lbl).sum())
        top = df[df['true_label']==true_lbl]['pred_leaf'].value_counts(normalize=True).head(5)
        print(f'\n  {true_lbl}  (n={n})')
        print(f'    HCE : ' + '  |  '.join([f'{p}: {v*100:.0f}%' for p, v in top.items()]))

# (1b) Glioma: lineage prediction only
print('\n' + '=' * 100)
print(f'  (1b) GLIOMA LINEAGE PREDICTION ({GLIOMA_DATASET})')
print('=' * 100)
if GLIOMA_DATASET in lab_results_hce:
    df_g = lab_results_hce[GLIOMA_DATASET]
    # Group by true label; show predicted lineage distribution
    for true_lbl in sorted(df_g['true_label'].unique()):
        n = int((df_g['true_label'] == true_lbl).sum())
        pred_lineages = df_g[df_g['true_label']==true_lbl]['pred_leaf'].map(assign_lineage)
        top = pred_lineages.value_counts(normalize=True).head(5)
        print(f'\n  {true_lbl}  (n={n})')
        print(f'    Predicted lineage: ' + '  |  '.join([f'{p}: {v*100:.0f}%' for p, v in top.items()]))

# (2) Lineage-match rate per dataset
print('\n' + '=' * 100)
print('  (2) LINEAGE-MATCH RATE')
print('=' * 100)
print(f'\n  {"Dataset":<26}{"HCE":>10}  Notes')
print('  ' + '-' * 50)
for lab_name in lab_results_hce.keys():
    df = lab_results_hce[lab_name]
    rate = (df['pred_leaf'].map(assign_lineage) == df['true_label'].map(assign_lineage)).mean()
    note = '(lineage-only; tumor states have no leaf equivalent)' if lab_name == GLIOMA_DATASET else ''
    print(f'  {lab_name:<26}{rate*100:>9.1f}%  {note}')

combined_all = pd.concat(list(lab_results_hce.values()), ignore_index=True)
pooled_all   = (combined_all['pred_leaf'].map(assign_lineage) == combined_all['true_label'].map(assign_lineage)).mean()
# Pooled excluding glioma
non_glioma = [v for k, v in lab_results_hce.items() if k != GLIOMA_DATASET]
pooled_non_glioma = None
if non_glioma:
    combined_ng = pd.concat(non_glioma, ignore_index=True)
    pooled_non_glioma = (combined_ng['pred_leaf'].map(assign_lineage) == combined_ng['true_label'].map(assign_lineage)).mean()
print(f'\n  POOLED all {len(lab_results_hce)} datasets       : {pooled_all*100:.1f}%')
if pooled_non_glioma is not None:
    print(f'  POOLED excl. glioma ({len(non_glioma)} datasets) : {pooled_non_glioma*100:.1f}%')

# (3) Attractor-concentration metric (v11)
# Max fraction of distinct true classes for which any single pred_leaf is top-1.
print('\n' + '=' * 100)
print('  (3) ATTRACTOR-CONCENTRATION METRIC (v11)')
print('=' * 100)
print(f'  = max over pred_leaves of: # distinct true classes where pred_leaf is top-1 / total true classes')
print(f'  Lower is better (was ~0.5 for committed-OPC in v10; target <0.15 after OPC merge).')
print()
for lab_name, df in lab_results_hce.items():
    true_classes = df['true_label'].unique()
    attractor_counts = Counter()
    for tc in true_classes:
        top1 = df[df['true_label']==tc]['pred_leaf'].value_counts().index[0]
        attractor_counts[top1] += 1
    total_true = len(true_classes)
    if total_true == 0:
        continue
    worst_leaf, worst_count = attractor_counts.most_common(1)[0]
    concentration = worst_count / total_true
    top3 = attractor_counts.most_common(3)
    print(f'  {lab_name}: concentration={concentration:.2f}  (worst: "{worst_leaf}" is top-1 for {worst_count}/{total_true} true classes)')
    print(f'    Top-3 attractors: ' + ', '.join([f'"{l}" ({c})' for l, c in top3]))

# (4) Ancestor recall @ k on in-distribution test set
def ancestor_set(node, ontology, max_depth=30):
    out, cur, depth = set(), ontology.get(node), 0
    while cur is not None and depth < max_depth and cur not in out:
        out.add(cur); cur = ontology.get(cur); depth += 1
    return out

def lca_hops(true_name, pred_name, ontology, max_k=30):
    if true_name == pred_name: return 0
    true_or_anc = {true_name} | ancestor_set(true_name, ontology, max_k)
    cur, k = pred_name, 0
    while cur is not None and k < max_k:
        if cur in true_or_anc: return k
        cur = ontology.get(cur); k += 1
    return None

def ancestor_recall_table(preds, trues):
    cache, depths = {}, []
    for t, p in zip(trues, preds):
        key = (int(t), int(p))
        if key not in cache:
            cache[key] = lca_hops(class_names[t], class_names[p], combined_ontology)
        depths.append(cache[key])
    arr = np.array([d if d is not None else 999 for d in depths])
    return {
        'leaf-exact (k=0)':  float((arr == 0).mean()),
        'within k<=1':       float((arr <= 1).mean()),
        'within k<=2':       float((arr <= 2).mean()),
        'within k<=3':       float((arr <= 3).mean()),
        'any shared anc':    float((arr <= 30).mean()),
    }

print('\n' + '=' * 100)
print('  (4) ANCESTOR RECALL @ k  (in-distribution test set)')
print('=' * 100)
tbl = ancestor_recall_table(hce_preds, hce_trues)
print(f'\n  {"Metric":<22}{"HCE":>10}')
print('  ' + '-' * 32)
for k, v in tbl.items():
    print(f'  {k:<22}{v*100:>9.2f}%')

print('\n  Per-organ ancestor-recall @ k<=2:')
print(f'  {"Organ":<20}{"HCE":>10}')
print('  ' + '-' * 30)
test_organ_np = np.asarray(test_organ)
for cfg in ORGAN_CONFIGS:
    mask = (test_organ_np == cfg['id'])
    if mask.sum() == 0: continue
    sub = ancestor_recall_table(hce_preds[mask], hce_trues[mask])
    print(f'  {cfg["name"]:<20}{sub["within k<=2"]*100:>9.2f}%')

print('\n[OK] Deep biological-accuracy analysis complete (v11)')


## 15. Confusion Matrix Visualizations

In [ ]:
import seaborn as sns

LINEAGE_ORDER = ['Myeloid','Lymphoid','Erythroid/Megakaryocyte','Hematopoietic Progenitor',
                 'Endothelial','Mesenchymal/Stromal','Epithelial','Glial','Neuronal',
                 'Tumor (glioma-like)','Other']

def lineage_confusion(df, order=LINEAGE_ORDER):
    tl = df['true_label'].map(assign_lineage)
    pl = df['pred_leaf'].map(assign_lineage)
    cm = pd.crosstab(tl, pl, normalize='index') * 100
    rows = [o for o in order if o in cm.index]
    cols = [o for o in order if o in cm.columns]
    return cm.reindex(index=rows, columns=cols).fillna(0)


# Per-dataset lineage heatmaps (HCE only)
n_ds = len(LAB_CONFIGS)
fig, axes = plt.subplots(n_ds, 1, figsize=(10, 4.5 * n_ds))
if n_ds == 1: axes = [axes]
fig.suptitle('HCE v11 Zero-Shot Lineage Confusion', fontsize=14, y=1.00)
for ax, lab_cfg in zip(axes, LAB_CONFIGS):
    name = lab_cfg['name']
    df = lab_results_hce[name]
    true_lin = df['true_label'].map(assign_lineage)
    acc      = (df['pred_leaf'].map(assign_lineage) == true_lin).mean() * 100
    cm = lineage_confusion(df)
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100, cbar=False,
                ax=ax, annot_kws={'size': 9}, linewidths=0.4, linecolor='lightgray')
    ax.set_title(f'{name}  (lineage match = {acc:.1f}%)', fontsize=11)
    ax.set_xlabel('Predicted lineage'); ax.set_ylabel('True lineage')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v11_cm_lineage_per_dataset.png'), dpi=150, bbox_inches='tight')
plt.show()


# Class-level heatmaps per dataset
TOP_N_PREDS = 15
for lab_cfg in LAB_CONFIGS:
    name = lab_cfg['name']
    df = lab_results_hce[name]
    cm = pd.crosstab(df['true_label'], df['pred_leaf'], normalize='index') * 100
    top_cols = cm.sum(axis=0).sort_values(ascending=False).head(TOP_N_PREDS).index.tolist()
    cm = cm.reindex(columns=top_cols, fill_value=0)
    h = max(4, 0.45 * len(cm.index) + 1.5)
    w = max(8, 0.55 * len(cm.columns) + 2.0)
    fig, ax = plt.subplots(figsize=(w, h))
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
                cbar_kws={'label': '% of true class'},
                ax=ax, annot_kws={'size': 7}, linewidths=0.3, linecolor='whitesmoke')
    ax.set_title(f'{name} — HCE class-level confusion (top {len(top_cols)} predicted classes)', fontsize=12)
    ax.set_xlabel('Predicted cell type'); ax.set_ylabel('True label')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    plt.tight_layout()
    safe = name.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(os.path.join(OUT_DIR, f'hce_v11_cm_classlevel_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.show()


# Combined pooled lineage matrix
combined_hce = pd.concat(list(lab_results_hce.values()), ignore_index=True)
true_lin_all = combined_hce['true_label'].map(assign_lineage)
acc_all      = (combined_hce['pred_leaf'].map(assign_lineage) == true_lin_all).mean() * 100

fig, ax = plt.subplots(figsize=(11, 8))
cm = lineage_confusion(combined_hce)
sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
            cbar_kws={'label': '% of true row'},
            ax=ax, annot_kws={'size': 10}, linewidths=0.5, linecolor='lightgray')
ax.set_title(f'HCE v11 — Combined Zero-Shot Lineage Confusion (all {len(LAB_CONFIGS)} lab datasets pooled)\noverall lineage match = {acc_all:.1f}%', fontsize=12)
ax.set_xlabel('Predicted lineage'); ax.set_ylabel('True lineage')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v11_cm_lineage_combined.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n[OK] Confusion matrix visualizations complete')

## 16. Final Summary

In [ ]:
print('=' * 80)
print('  HCE v11 — FINAL SUMMARY')
print('=' * 80)

print('\nTRAINING')
print(f'  Best validation accuracy : {hce_best_acc:.4f}')

print('\nTEST SET (in-distribution)')
m = results_hce['COMBINED']
print(f'  accuracy            : {m["accuracy"]:.4f}')
print(f'  macro_recall        : {m["macro_recall"]:.4f}')
print(f'  macro_f1            : {m["macro_f1"]:.4f}')
print(f'  zero_recall_count   : {m["zero_recall_count"]}')
print(f'  recall_50_pct       : {m["recall_50_pct"]:.4f}')
print(f'  recall_80_pct       : {m["recall_80_pct"]:.4f}')

print('\nPER-ORGAN MACRO RECALL (test set)')
print(f'  {"Organ":<20}{"HCE":>10}')
print('  ' + '-' * 30)
for cfg in ORGAN_CONFIGS:
    if cfg['name'] in results_hce:
        r = results_hce[cfg['name']]['macro_recall']
        print(f'  {cfg["name"]:<20}{r:>10.4f}')

print('\nZERO-SHOT LINEAGE MATCH (per lab dataset)')
print(f'  {"Dataset":<26}{"HCE":>10}')
print('  ' + '-' * 36)
for lab_name in lab_results_hce.keys():
    df = lab_results_hce[lab_name]
    rate = (df['pred_leaf'].map(assign_lineage) == df['true_label'].map(assign_lineage)).mean()
    print(f'  {lab_name:<26}{rate*100:>9.1f}%')
print(f'  {"POOLED":<26}{acc_all:>9.1f}%')

print('\n  Outputs saved to:', OUT_DIR)

## 18. Biology Spot-Checks (v8 acceptance criteria)


In [ ]:
# v11 biology spot-checks — HCE only
SPOT_CHECKS = [
    ('lymphoid',          'CD4_T_cell',     ['CD4'],                           'lineage'),
    ('lymphoid',          'Treg_cell',      ['CD4', 'regulatory'],             'lineage'),
    ('lymphoid',          'CD8_T_cell',     ['CD8'],                           'leaf'),
    ('myeloid',           'monocyte_',      ['monocyte'],                      'leaf'),
    ('Brain_normal',      'oligodendrocyte', ['oligodendrocyte'],              'leaf'),
    ('Liver_normal',      'hepatocyte',     ['hepatocyte'],                    'leaf'),
    ('Liver_normal',      'cholangiocyte',  ['cholangiocyte'],                 'leaf'),
    ('Lymph_node_normal', 'LN_stroma_cell', ['fibroblast', 'stroma'],          'lineage'),
    ('Brain_normal',      'GABAergic',      ['interneuron', 'inhibitory neuron','GABA'], 'lineage'),
]

def _eval_spot_checks(lab_results, model_label, has_lineage):
    """has_lineage=True for HCE (pred_lineage column exists), False for CE."""
    print(f'\n  --- Spot checks for {model_label} ---')
    print(f'  {"Dataset":<22} {"True label":<28} {"Top pred":<45} {"Axis":<8} {"PASS?":<6}')
    print('  ' + '-' * 110)
    n_pass, n_total = 0, 0
    for ds_name, true_sub, allowed_subs, axis in SPOT_CHECKS:
        df = lab_results.get(ds_name)
        if df is None:
            print(f'  {ds_name:<22}  (dataset not found)')
            continue
        if axis == 'lineage' and has_lineage:
            pred_col = 'pred_lineage'
        else:
            pred_col = 'pred_leaf'
        sub = df[df['true_label'].str.contains(true_sub, case=False, na=False, regex=False)]
        if len(sub) == 0:
            print(f'  {ds_name:<22} {true_sub:<28}  (no matching cells)')
            continue
        top = sub[pred_col].value_counts()
        top_pred = top.index[0]
        passed = any(a.lower() in top_pred.lower() for a in allowed_subs)
        n_total += 1; n_pass += int(passed)
        axis_used = axis if (axis=='leaf' or has_lineage) else 'leaf*'
        print(f'  {ds_name:<22} {true_sub:<28} {top_pred:<45} {axis_used:<8} {"PASS" if passed else "FAIL":<6}')
    print(f'\n  {model_label} spot checks: {n_pass}/{n_total}')
    return n_pass, n_total

hce_pass, hce_total = _eval_spot_checks(lab_results_hce, 'HCE v11', has_lineage=True)
print(f'\n  HCE spot checks: {hce_pass}/{hce_total}')
print('  (acceptance target: ≥ 6 of 9)')
